In [2]:
import numpy as np
import pandas as pd 
import os
import re
import ast
from collections import Counter
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns   
from sklearn.cluster import KMeans
from collections import defaultdict
import plotly.express as px
import plotly.graph_objects as go
from plotly.offline import iplot
import warnings
warnings.filterwarnings('ignore')

In [3]:
# --- ADIM 2: Yeni teknik özellikleri oku ---
df_ozellik = pd.read_csv(r"../web-scraping-data\data\raw\beko\beko_teknik_ozellik.csv")

# --- ADIM 3: Sütun adaylarını ve value_counts değerlerini çıkar ---
all_keys = []

for entry in df_ozellik['teknik_ozellikler']:
    try:
        # String halindeki dictionary yapısını Python sözlüğüne çeviriyoruz
        # Not: ast.literal_eval JSON kütüphanesine göre bu formatta daha esnektir
        d = ast.literal_eval(entry)
        all_keys.extend(d.keys())
    except:
        # Eğer veri bozuksa veya boşsa atla
        continue

# Tüm anahtarları bir Series yapıp value_counts alıyoruz
tech_column_candidates = pd.Series(all_keys).value_counts()

print("--- Teknik Özellik Sütun Adayları (Value Counts) ---")
print(tech_column_candidates.head(50)) # İlk 50 taneyi listeler

--- Teknik Özellik Sütun Adayları (Value Counts) ---
Ağırlık: Paketsiz                              33
Boyut (cm) (GxYxD)                             33
Derinlik                                       32
Ürün Rengi                                     15
Güç                                            12
Kapasite                                        9
23 cm                                           9
Otomatik Kapanma                                9
360° Dönebilen Kablosuz Kullanım                9
Kablo Sarma Yuvası                              9
Işıklı Açma/Kapama Düğmesi                      9
Ürün Tipi                                       8
Gövde Malzemesi                                 6
Rezistans Tipi                                  6
Renk                                            6
360° Taban                                      6
Şeffaf Su Seviye Göstergesi                     6
Tek Dokunuşla Açılabilen Kapak                  6
Kireç Filtresi                                 

In [4]:
def fix_key_value_swap(tech_str):
    try:
        tech_dict = ast.literal_eval(tech_str)
        cleaned_dict = {}
        
        # Sayı ile başlayan veya cm, kg, TL, gr, lt gibi birim içeren kelimeler için regex
        measurement_pattern = re.compile(r'^\d|cm|kg|TL|gr|lt|W|V|Hz|’', re.IGNORECASE)

        for key, value in tech_dict.items():
            # KEY ölçü birimi içeriyor ama VALUE içermiyorsa (Terslik varsa)
            if measurement_pattern.search(str(key)) and not measurement_pattern.search(str(value)):
                cleaned_dict[str(value).strip()] = str(key).strip()
            else:
                # Normal haliyle ekle
                cleaned_dict[str(key).strip()] = str(key).strip() # BURADA KÜÇÜK BİR HATA YAPMIŞIM: str(value).strip() olmalı
                
        # Düzeltilmiş hali:
        cleaned_dict = { (str(value).strip() if measurement_pattern.search(str(key)) and not measurement_pattern.search(str(value)) else str(key).strip()): 
                         (str(key).strip() if measurement_pattern.search(str(key)) and not measurement_pattern.search(str(value)) else str(value).strip()) 
                         for key, value in tech_dict.items() }
        
        return str(cleaned_dict)
    except:
        return tech_str

# 1. Temizleme fonksiyonunu elindeki df_ozellik'e uygula
df_ozellik['teknik_ozellikler_temiz'] = df_ozellik['teknik_ozellikler'].apply(fix_key_value_swap)

# 2. Kontrol: Yeni aday listesini tekrar hesapla
all_cleaned_keys = []
for entry in df_ozellik['teknik_ozellikler_temiz']:
    try:
        d = ast.literal_eval(entry)
        all_cleaned_keys.extend(d.keys())
    except:
        continue

new_candidates = pd.Series(all_cleaned_keys).value_counts()

print("--- TEMİZLİK SONRASI Sütun Adayları ---")
print(new_candidates.head(30))

--- TEMİZLİK SONRASI Sütun Adayları ---
Derinlik                                       39
Genişlik                                       39
Yükseklik                                      39
Ağırlık: Paketsiz                              33
Boyut (cm) (GxYxD)                             33
Ürün Rengi                                     15
Güç                                            12
Kapasite                                        9
Otomatik Kapanma                                9
360° Dönebilen Kablosuz Kullanım                9
Kablo Sarma Yuvası                              9
Işıklı Açma/Kapama Düğmesi                      9
Ürün Tipi                                       8
Paslanmaz Çelik                                 7
Rezistans Tipi                                  6
Renk                                            6
360° Taban                                      6
Şeffaf Su Seviye Göstergesi                     6
Tek Dokunuşla Açılabilen Kapak                  6
Kireç Filt

In [5]:
# 1. Değerleri toplamak için bir sözlük yapısı kuralım
value_map = defaultdict(set)

for entry in df_ozellik['teknik_ozellikler_temiz']:
    try:
        specs = ast.literal_eval(entry)
        for k, v in specs.items():
            value_map[k].add(v)
    except:
        continue

# 2. En popüler 15 aday için örnek değerleri yazdıralım
print("--- Teknik Sütunların Altındaki Örnek Değerler ---\n")
top_15_keys = new_candidates.head(15).index

for key in top_15_keys:
    # Her anahtar için ilk 5 eşsiz örneği gösterelim (karışıklık olmasın diye)
    examples = list(value_map[key])[:5]
    print(f"**{key}** : {examples}")

--- Teknik Sütunların Altındaki Örnek Değerler ---

**Derinlik** : ['29.8 cm', '15.6 cm', '6 cm', '32 cm', '60 cm']
**Genişlik** : ['9 cm', '61 cm (60 cm ile Aynı Alana Montaj Kolaylığı)', '16 cm', '25 cm', '19 cm']
**Yükseklik** : ['25 cm', '70 cm', '22 cm', '8 cm', '27 cm']
**Ağırlık: Paketsiz** : ['8.23 kg', '78 kg', '4.9 kg', '9.8 kg', '0.514 kg']
**Boyut (cm) (GxYxD)** : ['59.9 cm', '47.5 cm', '18.1 cm', '15.5 cm', '25 cm']
**Ürün Rengi** : ['İnoks', 'Gümüş', 'Beyaz', 'Siyah', 'Rose gold']
**Güç** : ['1500 W', '1900 W', '1650 W', '2200 W', '40 W']
**Kapasite** : ['1.2 L', '1.5 L', '10 kg', '50 L', '1.7 L']
**Otomatik Kapanma** : ['Var']
**360° Dönebilen Kablosuz Kullanım** : ['Var']
**Kablo Sarma Yuvası** : ['Var']
**Işıklı Açma/Kapama Düğmesi** : ['Var']
**Ürün Tipi** : ['Tam Ankastre', 'Seramik Isıtıcı', 'Solo', 'Tezgah Seviyesi', 'Tam Boy Su Pınarı']
**Paslanmaz Çelik** : ['Gövde Malzemesi', 'Çay Filtresi']
**Rezistans Tipi** : ['Gizli']


In [6]:
# 1. Sayıları ayıklayan temizlik fonksiyonu
def clean_to_numeric(val):
    if pd.isna(val) or val == '': return val
    # Sayısal değerleri çek (65 cm -> 65.0)
    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", str(val).replace(',', '.'))
    return float(numbers[0]) if numbers else val

# 2. Sütun Çeviri Haritası
translation_map = {
    'Derinlik': 'Depth', 'Genişlik': 'Width', 'Yükseklik': 'Height',
    'Ağırlık: Paketsiz': 'Weight_kg', 'Güç': 'Power_W', 'Kapasite': 'Capacity',
    'Otomatik Kapanma': 'Auto_Off', '360° Dönebilen Kablosuz Kullanım': 'Wireless_360',
    'Kablo Sarma Yuvası': 'Cable_Storage', 'Işıklı Açma/Kapama Düğmesi': 'Illuminated_Button',
    'Paslanmaz Çelik': 'Is_Stainless_Steel', 'Rezistans Tipi': 'Hidden_Heating',
    'Ürün Tipi': 'Product_Type', 'Ürün Rengi': 'Color'
}

def transform_columns_only(df):
    measurement_pattern = re.compile(r'^\d|cm|kg|TL|gr|lt|W|V|Hz|’', re.IGNORECASE)
    all_rows_specs = []
    
    # teknik_ozellikler_temiz veya teknik_ozellikler sütunundan hangisi varsa onu kullanalım
    source_col = 'teknik_ozellikler_temiz' if 'teknik_ozellikler_temiz' in df.columns else 'teknik_ozellikler'
    
    for idx, row in df.iterrows():
        try:
            specs_dict = ast.literal_eval(row[source_col])
            processed = {}
            for k, v in specs_dict.items():
                # Key-Value Swap düzeltmesi
                final_k, final_v = (v, k) if measurement_pattern.search(str(k)) and not measurement_pattern.search(str(v)) else (k, v)
                
                # İngilizce karşılığı varsa yeni sütun ismiyle ekle
                eng_k = translation_map.get(str(final_k).strip())
                if eng_k:
                    processed[eng_k] = clean_to_numeric(str(final_v).strip())
            all_rows_specs.append(processed)
        except:
            all_rows_specs.append({})

    df_specs = pd.DataFrame(all_rows_specs)
    # Mevcut df ile yeni sütunları yan yana getir
    return pd.concat([df.reset_index(drop=True), df_specs.reset_index(drop=True)], axis=1)

df_ozellik_cols = transform_columns_only(df_ozellik)

# Kontrol: İlk 5 satırda yeni sütunları görelim
print("--- Sütunlar Başarıyla Eklendi ve Temizlendi ---")
new_cols = [c for c in translation_map.values() if c in df_ozellik_cols.columns]
print(df_ozellik_cols[new_cols].head())

--- Sütunlar Başarıyla Eklendi ve Temizlendi ---
   Depth  Width  Height  Weight_kg  Power_W  Capacity Auto_Off Wireless_360  \
0   65.0   60.0    85.0      78.00      NaN       NaN      NaN          NaN   
1   60.0   60.0    85.0      71.00      NaN      10.0      NaN          NaN   
2   26.0   48.0     8.0       7.86      NaN       NaN      NaN          NaN   
3   38.2   51.0    30.0      14.50      NaN       NaN      NaN          NaN   
4   32.0   31.0    93.0      11.20      NaN       NaN      NaN          NaN   

  Cable_Storage Illuminated_Button Is_Stainless_Steel Hidden_Heating  \
0           NaN                NaN                NaN            NaN   
1           NaN                NaN                NaN            NaN   
2           NaN                NaN                NaN            NaN   
3           NaN                NaN                NaN            NaN   
4           NaN                NaN                NaN            NaN   

        Product_Type  Color  
0            

In [7]:
display(df_ozellik_cols[new_cols].head())

,Depth,Width,Height,Weight_kg,Power_W,Capacity,Auto_Off,Wireless_360,Cable_Storage,Illuminated_Button,Is_Stainless_Steel,Hidden_Heating,Product_Type,Color
0,65.0,60.0,85.0,78.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,60.0,60.0,85.0,71.00,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,Solo,Beyaz
2,26.0,48.0,8.0,7.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Solo,Beyaz
3,38.2,51.0,30.0,14.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Solo,Gümüş
4,32.0,31.0,93.0,11.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tam Boy Su Pınarı,Beyaz


In [8]:
display(df_ozellik_cols.head())

,urun_linki,urun_adi,fiyat,teknik_ozellikler,teknik_ozellikler_temiz,Depth,Width,Height,Weight_kg,Capacity,Color,Product_Type,Power_W,Is_Stainless_Steel,Hidden_Heating,Auto_Off,Wireless_360,Cable_Storage,Illuminated_Button
0,https://www.beko.com.tr/kurutma-makinesi/cmb-1...,CMB 1280 YKIG Kurutma Makinesi,48.479 TL,"{""65 cm"": ""Derinlik"", ""60 cm"": ""Genişlik"", ""85...","{'Derinlik': '65 cm', 'Genişlik': '60 cm', 'Yü...",65.0,60.0,85.0,78.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.beko.com.tr/kurutmali-camasir-maki...,CMX 1070 YK I Kurutmalı Çamaşır Makinesi,43.349 TL,"{""Yıka-kurut çeviriminde enerji verimlilik sın...",{'D': 'Yıka-kurut çeviriminde enerji verimlili...,60.0,60.0,85.0,71.00,10.0,Beyaz,Solo,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.beko.com.tr/set-ustu-ocak/bsoe-301...,BSOE 301 B Set Üstü Ocak,5.370 TL,"{""Ocak Tipi Ve Göz Sayısı"": ""2 Gözü Elektrikli...",{'Ocak Tipi Ve Göz Sayısı': '2 Gözü Elektrikli...,26.0,48.0,8.0,7.86,NaN,Beyaz,Solo,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.beko.com.tr/mikrodalga-firin/bmd-3...,BMD 310 DS Mikrodalga Fırın,11.217 TL,"{""Mikrodalga Kontrol Paneli"": ""Elektronik"", ""M...","{'Mikrodalga Kontrol Paneli': 'Elektronik', 'M...",38.2,51.0,30.0,14.50,NaN,Gümüş,Solo,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.beko.com.tr/su-sebili/5550-bu-su-s...,5550 BU Su Sebili,10.349 TL,"{""Damacana Yeri"": ""Üstten Yüklemeli"", ""Ürün Re...","{'Damacana Yeri': 'Üstten Yüklemeli', 'Ürün Re...",32.0,31.0,93.0,11.20,NaN,Beyaz,Tam Boy Su Pınarı,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
cols_to_drop = ['fiyat', 'teknik_ozellikler', 'teknik_ozellikler_temiz']

if 'df_ozellik_cols' in globals():
    df_ozellik_cols = df_ozellik_cols.drop(columns=[c for c in cols_to_drop if c in df_ozellik_cols.columns])

if 'df_ozellik' in globals():
    df_ozellik = df_ozellik.drop(columns=[c for c in cols_to_drop if c in df_ozellik.columns])

print("İlgili sütunlar silindi.")

İlgili sütunlar silindi.


In [10]:
display(df_ozellik_cols.head())

,urun_linki,urun_adi,Depth,Width,Height,Weight_kg,Capacity,Color,Product_Type,Power_W,Is_Stainless_Steel,Hidden_Heating,Auto_Off,Wireless_360,Cable_Storage,Illuminated_Button
0,https://www.beko.com.tr/kurutma-makinesi/cmb-1...,CMB 1280 YKIG Kurutma Makinesi,65.0,60.0,85.0,78.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.beko.com.tr/kurutmali-camasir-maki...,CMX 1070 YK I Kurutmalı Çamaşır Makinesi,60.0,60.0,85.0,71.00,10.0,Beyaz,Solo,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.beko.com.tr/set-ustu-ocak/bsoe-301...,BSOE 301 B Set Üstü Ocak,26.0,48.0,8.0,7.86,NaN,Beyaz,Solo,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.beko.com.tr/mikrodalga-firin/bmd-3...,BMD 310 DS Mikrodalga Fırın,38.2,51.0,30.0,14.50,NaN,Gümüş,Solo,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.beko.com.tr/su-sebili/5550-bu-su-s...,5550 BU Su Sebili,32.0,31.0,93.0,11.20,NaN,Beyaz,Tam Boy Su Pınarı,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df_ozellik_cols.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   urun_linki          120 non-null    str    
 1   urun_adi            120 non-null    str    
 2   Depth               39 non-null     float64
 3   Width               39 non-null     float64
 4   Height              39 non-null     float64
 5   Weight_kg           33 non-null     float64
 6   Capacity            9 non-null      float64
 7   Color               15 non-null     str    
 8   Product_Type        8 non-null      str    
 9   Power_W             12 non-null     float64
 10  Is_Stainless_Steel  7 non-null      str    
 11  Hidden_Heating      6 non-null      str    
 12  Auto_Off            9 non-null      str    
 13  Wireless_360        9 non-null      str    
 14  Cable_Storage       9 non-null      str    
 15  Illuminated_Button  9 non-null      str    
dtypes: float64(6), str(

In [12]:
# String / object tipindeki sütunlar için value_counts
string_cols = df_ozellik_cols.select_dtypes(include=["object", "string"]).columns

for col in string_cols:
    print(f"\n--- {col} ---")
    print(df_ozellik_cols[col].value_counts(dropna=False))


--- urun_linki ---
urun_linki
https://www.beko.com.tr/kurutma-makinesi/cmb-1280-ykig-kurutma-makinesi                        1
https://www.beko.com.tr/kurutmali-camasir-makinesi/cmx-1070-yk-i-kurutmali-camasir-makinesi    1
https://www.beko.com.tr/set-ustu-ocak/bsoe-301-b-set-ustu-ocak                                 1
https://www.beko.com.tr/mikrodalga-firin/bmd-310-ds-mikrodalga-firin                           1
https://www.beko.com.tr/su-sebili/5550-bu-su-sebili                                            1
                                                                                              ..
https://www.beko.com.tr/narenciye-sikacagi/ms-5410-b-narenciye-sikacagi                        1
https://www.beko.com.tr/cay-makinesi/bkk-2221-c-beko-dem-cay-makinesi-cay-makinesi             1
https://www.beko.com.tr/cay-makinesi/cm-5964-r-floral-cay-makinesi                             1
https://www.beko.com.tr/cay-makinesi/cm-5964-floral-cay-makinesi                               1

In [13]:
# 1. Product_Type Mapping (Eşleştirme Sözlüğü)
product_mapping = {
    'Tam Ankastre': 3,
    'Ankastre': 3,
    'Solo': 2,
    'Tezgah Seviyesi': 1
}

# Product_Type sütununu dönüştür (Eşleşmeyenleri 0 yap)
df_ozellik_cols['Product_Type'] = df_ozellik_cols['Product_Type'].map(product_mapping).fillna(0).astype(int)

# 2. Diğer Sütunlar İçin İkilik (Binary) Dönüşüm
# Değer varsa 1, NaN ise 0
binary_cols = [
    'Is_Stainless_Steel', 'Hidden_Heating', 'Auto_Off', 
    'Wireless_360', 'Cable_Storage', 'Illuminated_Button'
]

for col in binary_cols:
    df_ozellik_cols[col] = df_ozellik_cols[col].notna().astype(int)

# Sonuçları kontrol etmek için
print(df_ozellik_cols.head())

                                          urun_linki  \
0  https://www.beko.com.tr/kurutma-makinesi/cmb-1...   
1  https://www.beko.com.tr/kurutmali-camasir-maki...   
2  https://www.beko.com.tr/set-ustu-ocak/bsoe-301...   
3  https://www.beko.com.tr/mikrodalga-firin/bmd-3...   
4  https://www.beko.com.tr/su-sebili/5550-bu-su-s...   

                                   urun_adi  Depth  Width  Height  Weight_kg  \
0            CMB 1280 YKIG Kurutma Makinesi   65.0   60.0    85.0      78.00   
1  CMX 1070 YK I Kurutmalı Çamaşır Makinesi   60.0   60.0    85.0      71.00   
2                  BSOE 301 B Set Üstü Ocak   26.0   48.0     8.0       7.86   
3               BMD 310 DS Mikrodalga Fırın   38.2   51.0    30.0      14.50   
4                         5550 BU Su Sebili   32.0   31.0    93.0      11.20   

   Capacity  Color  Product_Type  Power_W  Is_Stainless_Steel  Hidden_Heating  \
0       NaN    NaN             0      NaN                   0               0   
1      10.0  Beyaz  

In [15]:
save_path = r"../web-scraping-data/data/claned/beko_temiz_eksik_ozellik.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
df_ozellik_cols.to_csv(save_path, index=False, encoding="utf-8-sig")
print(f"Kaydedildi: {save_path}")


Kaydedildi: ../web-scraping-data/data/claned/beko_temiz_eksik_ozellik.csv
